# 集成学习第四课：AdaBoost

## 本课目标

理解 Boosting 的第一种经典算法 AdaBoost：它如何通过“错题加权”让后面的模型重点学习前面没学好的样本。

一句话版：

```text
AdaBoost = 多个弱模型串行学习 + 错样本权重变大 + 模型加权投票
```

## 1. 从 Bagging 到 Boosting

前面学习的 Bagging 是并行思想：

```text
多个模型彼此独立训练，最后投票 / 平均。
```

Boosting 是串行思想：

```text
第 1 个模型先学。
第 2 个模型重点学第 1 个模型错的地方。
第 3 个模型继续补前面没学好的地方。
...
最后把多个模型组合起来。
```

所以 Boosting 的核心不是“大家各学各的”，而是“一个接一个补短板”。

## 2. AdaBoost 的名字

AdaBoost 是 Adaptive Boosting 的缩写。

Adaptive 的意思是“自适应”。

它自适应在哪里？

```text
上一轮分错的样本，下一轮权重会变大。
上一轮分对的样本，下一轮权重会相对变小。
```

这样后面的弱学习器会更关注难样本。

## 3. 什么是弱学习器？

弱学习器是指能力不强，但比乱猜好一点的模型。

AdaBoost 里常用的弱学习器是很浅的决策树，尤其是树桩 decision stump。

树桩就是深度为 1 的决策树：

```text
只问一个问题，就给出分类结果。
```

例如：

```text
如果 x1 <= 2.5，预测 A 类；否则预测 B 类。
```

单个树桩很弱，但很多树桩按顺序补错，就能组合成更强的模型。

## 4. AdaBoost 的核心流程

AdaBoost 可以粗略理解成下面 5 步：

1. 初始化所有样本权重相同。
2. 训练一个弱学习器。
3. 看它分错了哪些样本。
4. 提高分错样本的权重，降低分对样本的相对权重。
5. 重复多轮，最后让所有弱学习器加权投票。

直觉版：

```text
先做一遍题
-> 错题变重要
-> 下一轮重点做错题
-> 再错的题继续变重要
-> 最后综合每一轮老师的判断
```

## 4.1 数学符号准备

为了写清楚 AdaBoost，先把二分类标签写成：

$$
y_i \in \{-1, 1\}
$$

第 $t$ 轮训练出来的弱学习器记作：

$$
h_t(x)
$$

第 $i$ 个样本在第 $t$ 轮的权重记作：

$$
w_i^{(t)}
$$

初始时，所有样本一样重要。如果一共有 $m$ 个样本：

$$
w_i^{(1)} = \frac{1}{m}
$$

第 $t$ 个弱学习器的加权错误率记作：

$$
\varepsilon_t
$$

它说人话就是：

$$
\varepsilon_t
=
\text{第 }t\text{ 个弱学习器分错的样本权重之和}
$$

所以我们可以直接写成：

$$
\varepsilon_t
=
\sum_{h_t(x_i)\ne y_i}
w_i^{(t)}
$$

这个公式的意思非常简单：

$$
\text{普通错误率看错了几个样本}
$$

而 AdaBoost 的加权错误率看的是：

$$
\text{错掉的样本有多重要}
$$

如果第 $i$ 个样本被分错：

$$
h_t(x_i)\ne y_i
$$

那么它的权重会被加进错误率：

$$
w_i^{(t)}
$$

如果第 $i$ 个样本被分对：

$$
h_t(x_i)=y_i
$$

那么它不会贡献错误率：

$$
0
$$

也可以写成分情况：

$$
\text{第 }i\text{ 个样本对 }\varepsilon_t\text{ 的贡献}
=
\begin{cases}
w_i^{(t)}, & h_t(x_i)\ne y_i \\[6pt]
0, & h_t(x_i)=y_i
\end{cases}
$$

举个例子，假设有 $4$ 个样本，当前权重是：

$$
0.1,\quad 0.2,\quad 0.3,\quad 0.4
$$

如果弱学习器分错了第 $2$ 个和第 $4$ 个样本，那么：

$$
\varepsilon_t
=
0.2 + 0.4
=
0.6
$$

如果它分错的是第 $1$ 个和第 $2$ 个样本，那么：

$$
\varepsilon_t
=
0.1 + 0.2
=
0.3
$$

所以，加权错误率不是看“错了多少个”，而是看：

$$
\text{错的这些样本重不重要}
$$

## 4.2 弱学习器权重 $\alpha_t$

AdaBoost 会给每个弱学习器一个投票权重，记作：

$$
\alpha_t
$$

这个权重应该满足一个很自然的要求：

```text
错误率越低，说明这个弱学习器越可靠，投票声音应该越大。
错误率越接近 0.5，说明它越像随机猜，投票声音应该越小。
```

设第 $t$ 个弱学习器的加权错误率是：

$$
\varepsilon_t
$$

AdaBoost 最终使用的弱学习器权重是：

$$
\alpha_t = \frac{1}{2}\ln\frac{1-\varepsilon_t}{\varepsilon_t}
$$

先看这个公式的方向感：

- 如果 $\varepsilon_t = 0.1$，错误率很低，$\alpha_t$ 比较大。
- 如果 $\varepsilon_t = 0.45$，错误率接近乱猜，$\alpha_t$ 很小。
- 如果 $\varepsilon_t = 0.5$，它和乱猜差不多，$\alpha_t = 0$。

举几个数：

In [ ]:
import numpy as np

errors = np.array([0.1, 0.2, 1/3, 0.45, 0.49])
alphas = 0.5 * np.log((1 - errors) / errors)

list(zip(errors, alphas))

## 4.3 样本权重更新公式

AdaBoost 的样本权重更新可以直接写成分情况形式：

$$
w_i^{(t+1)}
=
\begin{cases}
\dfrac{w_i^{(t)} e^{-\alpha_t}}{Z_t},
& h_t(x_i)=y_i \\[10pt]
\dfrac{w_i^{(t)} e^{\alpha_t}}{Z_t},
& h_t(x_i)\ne y_i
\end{cases}
$$

这个公式就看两行：

如果第 $i$ 个样本被第 $t$ 个弱学习器分对了：

$$
h_t(x_i)=y_i
$$

那么：

$$
w_i^{(t+1)}
=
\frac{w_i^{(t)} e^{-\alpha_t}}{Z_t}
$$

因为：

$$
e^{-\alpha_t}<1
$$

所以分对样本的权重会变小。

如果第 $i$ 个样本被第 $t$ 个弱学习器分错了：

$$
h_t(x_i)\ne y_i
$$

那么：

$$
w_i^{(t+1)}
=
\frac{w_i^{(t)} e^{\alpha_t}}{Z_t}
$$

因为：

$$
e^{\alpha_t}>1
$$

所以分错样本的权重会变大。

这里的 $Z_t$ 只是归一化因子，用来保证更新后的所有样本权重加起来仍然等于 $1$。

所以这一节先记住一句话：

$$
\text{分对权重变小，分错权重变大。}
$$


## 4.3.1 $Z_t$ 是怎么来的？

$Z_t$ 是归一化因子。

说人话就是：

$$
Z_t
=
\text{所有未归一化新权重的总和}
$$

为什么需要它？

因为 AdaBoost 更新权重时，会先做两件事：

$$
\text{分对样本权重乘 } e^{-\alpha_t}
$$

$$
\text{分错样本权重乘 } e^{\alpha_t}
$$

这样一乘之后，所有样本的新权重加起来通常就不再等于 $1$。

但是样本权重必须满足：

$$
\sum_{i=1}^{m} w_i^{(t+1)} = 1
$$

所以要除以一个总数，把它们重新调整回总和为 $1$。这个总数就是 $Z_t$。

按照分情况写法，分对样本的未归一化权重之和是：

$$
\sum_{h_t(x_i)=y_i}
w_i^{(t)} e^{-\alpha_t}
$$

分错样本的未归一化权重之和是：

$$
\sum_{h_t(x_i)\ne y_i}
w_i^{(t)} e^{\alpha_t}
$$

所以：

$$
Z_t
=
\sum_{h_t(x_i)=y_i}
w_i^{(t)} e^{-\alpha_t}
+
\sum_{h_t(x_i)\ne y_i}
w_i^{(t)} e^{\alpha_t}
$$

举个小例子。

假设有 $3$ 个样本，旧权重是：

$$
0.2,\quad 0.3,\quad 0.5
$$

某个弱学习器把前两个样本分对，把第三个样本分错。

再假设：

$$
e^{-\alpha_t}=0.5
$$

$$
e^{\alpha_t}=2
$$

那么未归一化的新权重是：

$$
0.2\times 0.5 = 0.1
$$

$$
0.3\times 0.5 = 0.15
$$

$$
0.5\times 2 = 1.0
$$

它们加起来就是：

$$
Z_t
=
0.1 + 0.15 + 1.0
=
1.25
$$

所以真正的新权重是：

$$
\frac{0.1}{1.25}=0.08
$$

$$
\frac{0.15}{1.25}=0.12
$$

$$
\frac{1.0}{1.25}=0.8
$$

最后检查：

$$
0.08+0.12+0.8=1
$$

所以 $Z_t$ 不是凭空来的，它就是：

$$
\text{把权重放大或缩小之后，为了重新归一化而除掉的总和}
$$

## 4.4 $\alpha_t$ 公式怎么推出来？

这一段慢慢推。

第 $t$ 轮弱学习器的加权错误率是：

$$
\varepsilon_t
$$

那分对样本的总权重就是：

$$
1-\varepsilon_t
$$

更新权重时：

- 分对样本乘 $e^{-\alpha_t}$
- 分错样本乘 $e^{\alpha_t}$

所以归一化之前，所有样本的新权重总和是：

$$
Z_t =
(1-\varepsilon_t)e^{-\alpha_t}
+ \varepsilon_t e^{\alpha_t}
$$

AdaBoost 希望这个 $Z_t$ 尽量小。你可以先把它理解成：这一轮模型留下来的“总压力”越小越好。

为了推得更清楚，令：

$$
a = e^{\alpha_t}
$$

那么：

$$
e^{-\alpha_t} = \frac{1}{a}
$$

于是 $Z_t$ 可以改写成：

$$
Z_t =
\frac{1-\varepsilon_t}{a}
+ \varepsilon_t a
$$

现在只要找一个合适的 $a$，让 $Z_t$ 最小。

对 $a$ 求导：

$$
\frac{dZ_t}{da}
=
-\frac{1-\varepsilon_t}{a^2}
+ \varepsilon_t
$$

令导数等于 0：

$$
-\frac{1-\varepsilon_t}{a^2}
+ \varepsilon_t
= 0
$$

移项：

$$
\varepsilon_t
=
\frac{1-\varepsilon_t}{a^2}
$$

两边乘以 $a^2$：

$$
\varepsilon_t a^2
=
1-\varepsilon_t
$$

所以：

$$
a^2
=
\frac{1-\varepsilon_t}{\varepsilon_t}
$$

两边开方：

$$
a
=
\sqrt{\frac{1-\varepsilon_t}{\varepsilon_t}}
$$

但前面定义过：

$$
a = e^{\alpha_t}
$$

所以：

$$
e^{\alpha_t}
=
\sqrt{\frac{1-\varepsilon_t}{\varepsilon_t}}
$$

两边取自然对数：

$$
\alpha_t
=
\ln \sqrt{\frac{1-\varepsilon_t}{\varepsilon_t}}
$$

而：

$$
\ln \sqrt{x} = \frac{1}{2}\ln x
$$

因此：

$$
\alpha_t
=
\frac{1}{2}\ln\frac{1-\varepsilon_t}{\varepsilon_t}
$$

这就是 AdaBoost 弱学习器投票权重的公式。

## 4.5 完整推导例子：弱学习器从哪里来？

前面的手算例子如果直接给出每一轮预测，会少掉最关键的一步：

$$
\text{弱学习器是怎么选出来的？}
$$

下面重新做一个完整例子。

我们用一个一维数据集：

| 样本 | 特征 $x_i$ | 标签 $y_i$ |
|---:|---:|---:|
| $1$ | $1$ | $1$ |
| $2$ | $2$ | $1$ |
| $3$ | $3$ | $-1$ |
| $4$ | $4$ | $1$ |
| $5$ | $5$ | $-1$ |
| $6$ | $6$ | $-1$ |

弱学习器使用最简单的决策树桩，也就是只切一刀。

候选规则有两种方向。

第一种方向：

$$
G(x)
=
\begin{cases}
1, & x \le c \\[6pt]
-1, & x > c
\end{cases}
$$

第二种方向：

$$
G(x)
=
\begin{cases}
-1, & x \le c \\[6pt]
1, & x > c
\end{cases}
$$

这里的 $c$ 是阈值。

因为样本的特征是：

$$
1,\ 2,\ 3,\ 4,\ 5,\ 6
$$

所以候选阈值可以取相邻样本中间：

$$
0.5,\ 1.5,\ 2.5,\ 3.5,\ 4.5,\ 5.5
$$

AdaBoost 每一轮做的事情就是：

$$
\text{在这些候选树桩里，选加权错误率最小的那个。}
$$

### 第 $1$ 轮：选择第一个弱学习器

初始时所有样本权重相同：

$$
w_i^{(1)}=\frac{1}{6}
$$

因此第 $1$ 轮的加权错误率就是：

$$
\text{分错样本的权重之和}
$$

下面枚举候选树桩。

| 候选规则 | 分错样本 | 加权错误率 |
|---|---|---:|
| $x\le 0.5 \Rightarrow 1,\ x>0.5 \Rightarrow -1$ | $1,2,4$ | $\frac{3}{6}$ |
| $x\le 0.5 \Rightarrow -1,\ x>0.5 \Rightarrow 1$ | $3,5,6$ | $\frac{3}{6}$ |
| $x\le 1.5 \Rightarrow 1,\ x>1.5 \Rightarrow -1$ | $2,4$ | $\frac{2}{6}$ |
| $x\le 1.5 \Rightarrow -1,\ x>1.5 \Rightarrow 1$ | $1,3,5,6$ | $\frac{4}{6}$ |
| $x\le 2.5 \Rightarrow 1,\ x>2.5 \Rightarrow -1$ | $4$ | $\frac{1}{6}$ |
| $x\le 2.5 \Rightarrow -1,\ x>2.5 \Rightarrow 1$ | $1,2,3,5,6$ | $\frac{5}{6}$ |
| $x\le 3.5 \Rightarrow 1,\ x>3.5 \Rightarrow -1$ | $3,4$ | $\frac{2}{6}$ |
| $x\le 3.5 \Rightarrow -1,\ x>3.5 \Rightarrow 1$ | $1,2,5,6$ | $\frac{4}{6}$ |
| $x\le 4.5 \Rightarrow 1,\ x>4.5 \Rightarrow -1$ | $3$ | $\frac{1}{6}$ |
| $x\le 4.5 \Rightarrow -1,\ x>4.5 \Rightarrow 1$ | $1,2,4,5,6$ | $\frac{5}{6}$ |
| $x\le 5.5 \Rightarrow 1,\ x>5.5 \Rightarrow -1$ | $3,5$ | $\frac{2}{6}$ |
| $x\le 5.5 \Rightarrow -1,\ x>5.5 \Rightarrow 1$ | $1,2,4,6$ | $\frac{4}{6}$ |

最小加权错误率是：

$$
\frac{1}{6}
$$

有两个候选规则都能达到这个错误率。这里根据奥卡姆剃刀原理选择：

$$
G_1(x)
=
\begin{cases}
1, & x\le 2.5 \\[6pt]
-1, & x>2.5
\end{cases}
$$

于是第 $1$ 轮预测是：

| 样本 | $x_i$ | $y_i$ | $G_1(x_i)$ | 是否分错 |
|---:|---:|---:|---:|---|
| $1$ | $1$ | $1$ | $1$ | 否 |
| $2$ | $2$ | $1$ | $1$ | 否 |
| $3$ | $3$ | $-1$ | $-1$ | 否 |
| $4$ | $4$ | $1$ | $-1$ | 是 |
| $5$ | $5$ | $-1$ | $-1$ | 否 |
| $6$ | $6$ | $-1$ | $-1$ | 否 |

所以：

$$
\varepsilon_1=\frac{1}{6}
$$

弱学习器权重：

$$
\alpha_1
=
\frac{1}{2}\ln\frac{1-\frac{1}{6}}{\frac{1}{6}}
=
\frac{1}{2}\ln 5
\approx 0.8047
$$

因此：

$$
e^{-\alpha_1}
=
\frac{1}{\sqrt{5}}
\approx 0.4472
$$

$$
e^{\alpha_1}
=
\sqrt{5}
\approx 2.2361
$$

未归一化的新权重：

分对样本：

$$
\frac{1}{6}\cdot \frac{1}{\sqrt{5}}
$$

分错样本：

$$
\frac{1}{6}\cdot \sqrt{5}
$$

第 $1$ 轮有 $5$ 个样本分对，$1$ 个样本分错，所以：

$$
Z_1
=
5\cdot \frac{1}{6\sqrt{5}}
+
1\cdot \frac{\sqrt{5}}{6}
=
\frac{\sqrt{5}}{3}
\approx 0.7454
$$

归一化后：

分对样本权重变成：

$$
\frac{
\frac{1}{6\sqrt{5}}
}{
\frac{\sqrt{5}}{3}
}
=
\frac{1}{10}
$$

分错样本权重变成：

$$
\frac{
\frac{\sqrt{5}}{6}
}{
\frac{\sqrt{5}}{3}
}
=
\frac{1}{2}
$$

所以第 $1$ 轮之后的样本权重是：

$$
\left[
\frac{1}{10},
\frac{1}{10},
\frac{1}{10},
\frac{1}{2},
\frac{1}{10},
\frac{1}{10}
\right]
$$

### 第 $2$ 轮：用新权重选择第二个弱学习器

第 $2$ 轮开始时，样本权重是：

$$
\left[
\frac{1}{10},
\frac{1}{10},
\frac{1}{10},
\frac{1}{2},
\frac{1}{10},
\frac{1}{10}
\right]
$$

继续枚举候选树桩，选择加权错误率最小的规则。

这一轮选到：

$$
G_2(x)
=
\begin{cases}
1, & x\le 4.5 \\[6pt]
-1, & x>4.5
\end{cases}
$$

它的预测结果是：

| 样本 | $x_i$ | $y_i$ | $G_2(x_i)$ | 是否分错 | 当前权重 |
|---:|---:|---:|---:|---|---:|
| $1$ | $1$ | $1$ | $1$ | 否 | $\frac{1}{10}$ |
| $2$ | $2$ | $1$ | $1$ | 否 | $\frac{1}{10}$ |
| $3$ | $3$ | $-1$ | $1$ | 是 | $\frac{1}{10}$ |
| $4$ | $4$ | $1$ | $1$ | 否 | $\frac{1}{2}$ |
| $5$ | $5$ | $-1$ | $-1$ | 否 | $\frac{1}{10}$ |
| $6$ | $6$ | $-1$ | $-1$ | 否 | $\frac{1}{10}$ |

所以：

$$
\varepsilon_2=\frac{1}{10}
$$

弱学习器权重：

$$
\alpha_2
=
\frac{1}{2}\ln\frac{1-\frac{1}{10}}{\frac{1}{10}}
=
\frac{1}{2}\ln 9
=
\ln 3
\approx 1.0986
$$

因此：

$$
e^{-\alpha_2}
=
\frac{1}{3}
$$

$$
e^{\alpha_2}
=
3
$$

归一化因子：

$$
Z_2
=
\frac{9}{10}\cdot \frac{1}{3}
+
\frac{1}{10}\cdot 3
=
\frac{3}{5}
=
0.6
$$

归一化后，第 $2$ 轮之后的样本权重是：

$$
\left[
\frac{1}{18},
\frac{1}{18},
\frac{1}{2},
\frac{5}{18},
\frac{1}{18},
\frac{1}{18}
\right]
$$

注意：第 $2$ 轮分错的是第 $3$ 个样本，所以它的权重从：

$$
\frac{1}{10}
$$

上升到：

$$
\frac{1}{2}
$$

### 第 $3$ 轮：继续用新权重选择第三个弱学习器

第 $3$ 轮开始时，样本权重是：

$$
\left[
\frac{1}{18},
\frac{1}{18},
\frac{1}{2},
\frac{5}{18},
\frac{1}{18},
\frac{1}{18}
\right]
$$

继续枚举候选树桩，选择加权错误率最小的规则。

这一轮选到：

$$
G_3(x)
=
\begin{cases}
-1, & x\le 3.5 \\[6pt]
1, & x>3.5
\end{cases}
$$

它的预测结果是：

| 样本 | $x_i$ | $y_i$ | $G_3(x_i)$ | 是否分错 | 当前权重 |
|---:|---:|---:|---:|---|---:|
| $1$ | $1$ | $1$ | $-1$ | 是 | $\frac{1}{18}$ |
| $2$ | $2$ | $1$ | $-1$ | 是 | $\frac{1}{18}$ |
| $3$ | $3$ | $-1$ | $-1$ | 否 | $\frac{1}{2}$ |
| $4$ | $4$ | $1$ | $1$ | 否 | $\frac{5}{18}$ |
| $5$ | $5$ | $-1$ | $1$ | 是 | $\frac{1}{18}$ |
| $6$ | $6$ | $-1$ | $1$ | 是 | $\frac{1}{18}$ |

所以：

$$
\varepsilon_3
=
\frac{1}{18}
+
\frac{1}{18}
+
\frac{1}{18}
+
\frac{1}{18}
=
\frac{2}{9}
$$

弱学习器权重：

$$
\alpha_3
=
\frac{1}{2}\ln\frac{1-\frac{2}{9}}{\frac{2}{9}}
=
\frac{1}{2}\ln\frac{7}{2}
\approx 0.6264
$$

因此：

$$
e^{-\alpha_3}
=
\sqrt{\frac{2}{7}}
\approx 0.5345
$$

$$
e^{\alpha_3}
=
\sqrt{\frac{7}{2}}
\approx 1.8708
$$

归一化因子：

$$
Z_3
=
\frac{7}{9}\sqrt{\frac{2}{7}}
+
\frac{2}{9}\sqrt{\frac{7}{2}}
\approx 0.8315
$$

归一化后，第 $3$ 轮之后的样本权重是：

$$
\left[
\frac{1}{8},
\frac{1}{8},
\frac{9}{28},
\frac{5}{28},
\frac{1}{8},
\frac{1}{8}
\right]
$$

In [ ]:
import numpy as np
import pandas as pd

x = np.array([1, 2, 3, 4, 5, 6])
y = np.array([1, 1, -1, 1, -1, -1])

def stump_predict(x, threshold, direction):
    if direction == 1:
        return np.where(x <= threshold, 1, -1)
    return np.where(x <= threshold, -1, 1)

def all_stumps(x, y, weights):
    thresholds = [0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
    rows = []
    for threshold in thresholds:
        for direction in [1, -1]:
            pred = stump_predict(x, threshold, direction)
            wrong = pred != y
            error = weights[wrong].sum()
            if direction == 1:
                rule = f"x <= {threshold} -> 1, x > {threshold} -> -1"
            else:
                rule = f"x <= {threshold} -> -1, x > {threshold} -> 1"
            rows.append({
                "规则": rule,
                "分错样本": list(np.where(wrong)[0] + 1),
                "加权错误率": error,
                "threshold": threshold,
                "direction": direction,
                "prediction": pred,
            })
    return pd.DataFrame(rows).sort_values("加权错误率").reset_index(drop=True)

weights = np.ones(6) / 6
round_summaries = []
detail_tables = []

for round_index in range(1, 4):
    candidates = all_stumps(x, y, weights)
    best = candidates.iloc[0]
    pred = best["prediction"]
    wrong = pred != y
    error = best["加权错误率"]
    alpha = 0.5 * np.log((1 - error) / error)

    multiplier = np.where(wrong, np.exp(alpha), np.exp(-alpha))
    unnormalized = weights * multiplier
    z_t = unnormalized.sum()
    next_weights = unnormalized / z_t

    round_summaries.append({
        "轮数": round_index,
        "选择规则": best["规则"],
        "分错样本": best["分错样本"],
        "加权错误率": error,
        "alpha": alpha,
        "Z_t": z_t,
    })

    detail_tables.append(pd.DataFrame({
        "轮数": round_index,
        "样本": np.arange(1, 7),
        "x": x,
        "y": y,
        "预测": pred,
        "是否分错": wrong,
        "本轮权重": weights,
        "更新倍数": multiplier,
        "未归一化新权重": unnormalized,
        "归一化后新权重": next_weights,
    }))

    weights = next_weights

summary_table = pd.DataFrame(round_summaries)
detail_table = pd.concat(detail_tables, ignore_index=True)

summary_table, detail_table

### 三轮之后怎么预测？

三轮得到的弱学习器是：

$$
G_1(x)
=
\begin{cases}
1, & x\le 2.5 \\[6pt]
-1, & x>2.5
\end{cases}
$$

$$
G_2(x)
=
\begin{cases}
1, & x\le 4.5 \\[6pt]
-1, & x>4.5
\end{cases}
$$

$$
G_3(x)
=
\begin{cases}
-1, & x\le 3.5 \\[6pt]
1, & x>3.5
\end{cases}
$$

对应的投票权重是：

$$
\alpha_1\approx 0.8047
$$

$$
\alpha_2\approx 1.0986
$$

$$
\alpha_3\approx 0.6264
$$

对一个新样本 $x$，先计算加权投票分数：

$$
S(x)
=
\alpha_1G_1(x)
+
\alpha_2G_2(x)
+
\alpha_3G_3(x)
$$

最终分类写成分情况：

$$
H(x)
=
\begin{cases}
1, & S(x)>0 \\[6pt]
-1, & S(x)<0
\end{cases}
$$

这就是完整过程：

$$
\text{枚举弱学习器}
\rightarrow
\text{选加权错误率最小的弱学习器}
\rightarrow
\text{计算 }\alpha_t
\rightarrow
\text{更新并归一化样本权重}
\rightarrow
\text{进入下一轮}
$$

## 5. 弱学习器也有权重

AdaBoost 不只是给样本加权，也会给每个弱学习器一个权重。

直觉是：

```text
表现好的弱学习器，最终投票声音更大。
表现差的弱学习器，最终投票声音更小。
```

如果某个弱学习器错误率很低，它就更可信；如果错误率接近乱猜，它就不该有太大发言权。

In [ ]:
# AdaBoost 中二分类弱学习器权重的简化计算
# error 越小，alpha 越大；error 越接近 0.5，alpha 越接近 0
errors = np.array([0.1, 0.2, 0.3, 0.45])
alphas = 0.5 * np.log((1 - errors) / errors)

list(zip(errors, alphas))

上面的 `alpha` 可以先理解成“投票权重”。

错误率越低，投票权重越大。

## 6. 加权投票

Bagging 和随机森林通常是多数投票：

$$
\text{谁票数多，谁赢。}
$$

AdaBoost 不是普通投票，而是加权投票：

$$
\text{谁的投票权重总和大，谁赢。}
$$

每个弱学习器都有一个投票权重：

$$
\alpha_t
$$

错误率低的弱学习器更可靠，所以：

$$
\alpha_t\text{ 更大}
$$

错误率高的弱学习器不那么可靠，所以：

$$
\alpha_t\text{ 更小}
$$

假设有 $3$ 个弱学习器：

$$
G_1(x),\quad G_2(x),\quad G_3(x)
$$

它们的投票权重分别是：

$$
\alpha_1=0.8,\quad \alpha_2=0.5,\quad \alpha_3=0.3
$$

对某个样本 $x$，三个弱学习器预测：

$$
G_1(x)=1
$$

$$
G_2(x)=-1
$$

$$
G_3(x)=1
$$

普通投票看票数：

$$
1,\quad -1,\quad 1
$$

所以普通投票会选：

$$
1
$$

AdaBoost 看的是权重。

支持 $1$ 的弱学习器是 $G_1$ 和 $G_3$，所以支持 $1$ 的总权重是：

$$
A_+(x)
=
0.8+0.3
=
1.1
$$

支持 $-1$ 的弱学习器是 $G_2$，所以支持 $-1$ 的总权重是：

$$
A_-(x)
=
0.5
$$

因为：

$$
A_+(x)>A_-(x)
$$

所以最终预测：

$$
H(x)=1
$$

再看一个更能说明问题的例子。

如果三个弱学习器预测：

$$
G_1(x)=1
$$

$$
G_2(x)=-1
$$

$$
G_3(x)=-1
$$

普通投票会选：

$$
-1
$$

因为 $-1$ 有两票。

但如果投票权重是：

$$
\alpha_1=1.2,\quad \alpha_2=0.5,\quad \alpha_3=0.3
$$

支持 $1$ 的总权重是：

$$
A_+(x)=1.2
$$

支持 $-1$ 的总权重是：

$$
A_-(x)=0.5+0.3=0.8
$$

因为：

$$
A_+(x)>A_-(x)
$$

所以 AdaBoost 最终仍然预测：

$$
H(x)=1
$$

也就是说：

$$
\text{AdaBoost 不是看哪一类票数多，而是看哪一类票的总权重大。}
$$

一般写法是：

支持 $1$ 的总权重：

$$
A_+(x)
=
\sum_{G_t(x)=1}
\alpha_t
$$

支持 $-1$ 的总权重：

$$
A_-(x)
=
\sum_{G_t(x)=-1}
\alpha_t
$$

最终预测写成分情况：

$$
H(x)
=
\begin{cases}
1, & A_+(x)>A_-(x) \\[6pt]
-1, & A_+(x)<A_-(x)
\end{cases}
$$

In [ ]:
# AdaBoost 加权投票演示
import numpy as np

predictions = np.array([1, -1, -1])
alphas = np.array([1.2, 0.5, 0.3])

weight_for_positive = alphas[predictions == 1].sum()
weight_for_negative = alphas[predictions == -1].sum()

if weight_for_positive > weight_for_negative:
    final_prediction = 1
elif weight_for_positive < weight_for_negative:
    final_prediction = -1
else:
    final_prediction = 0

weight_for_positive, weight_for_negative, final_prediction

## 7. sklearn 实战：AdaBoost 分类

下面用 `make_moons` 数据集比较：

- 单个树桩
- AdaBoost + 多个树桩
- 随机森林

这里随机森林只是作为参照，不是说它一定总赢或总输。

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score

X, y = make_moons(n_samples=600, noise=0.3, random_state=22)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=22, stratify=y
)

stump = DecisionTreeClassifier(max_depth=1, random_state=22)
stump.fit(X_train, y_train)

try:
    adaboost = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=22),
        n_estimators=100,
        learning_rate=0.5,
        random_state=22,
    )
except TypeError:
    adaboost = AdaBoostClassifier(
        base_estimator=DecisionTreeClassifier(max_depth=1, random_state=22),
        n_estimators=100,
        learning_rate=0.5,
        random_state=22,
    )

adaboost.fit(X_train, y_train)

forest = RandomForestClassifier(
    n_estimators=100,
    random_state=22,
    n_jobs=-1,
)
forest.fit(X_train, y_train)

scores = {
    "单个树桩": accuracy_score(y_test, stump.predict(X_test)),
    "AdaBoost": accuracy_score(y_test, adaboost.predict(X_test)),
    "随机森林": accuracy_score(y_test, forest.predict(X_test)),
}

scores

通常可以看到，单个树桩能力很弱，但多个树桩通过 AdaBoost 串行组合后，效果会明显提升。

## 8. 查看每轮弱学习器权重

训练好的 AdaBoost 模型里，可以查看每个弱学习器的权重：

```python
model.estimator_weights_
```

也可以查看每轮弱学习器的错误率：

```python
model.estimator_errors_
```

In [ ]:
first_10_weights = adaboost.estimator_weights_[:10]
first_10_errors = adaboost.estimator_errors_[:10]

list(zip(first_10_errors, first_10_weights))

错误率越低的弱学习器，通常权重越大。

这和前面讲的加权投票直觉是一致的。

## 9. `n_estimators` 和 `learning_rate`

AdaBoost 里最重要的两个参数：

| 参数 | 含义 |
|---|---|
| `n_estimators` | 弱学习器数量，也就是 Boosting 轮数 |
| `learning_rate` | 学习率，通常记作 $\nu$ |

`n_estimators` 控制一共有多少个弱学习器：

$$
G_1(x),\ G_2(x),\ \dots,\ G_T(x)
$$

其中：

$$
T=\text{n\_estimators}
$$

`learning_rate` 控制每一轮弱学习器的影响有多大。

原始 AdaBoost 中，第 $t$ 个弱学习器的投票权重是：

$$
\alpha_t
$$

加入学习率之后，实际使用的是：

$$
\nu\alpha_t
$$

也就是说，学习率会把每一轮弱学习器的声音缩小。

如果：

$$
\nu=1
$$

那就是原始 AdaBoost。

如果：

$$
\nu=0.5
$$

那么第 $t$ 个弱学习器实际投票权重变成：

$$
0.5\alpha_t
$$

### 学习率在最终投票里的体现

不加学习率时，加权投票看：

$$
A_+(x)
=
\sum_{G_t(x)=1}
\alpha_t
$$

$$
A_-(x)
=
\sum_{G_t(x)=-1}
\alpha_t
$$

加入学习率后变成：

$$
A_+(x)
=
\sum_{G_t(x)=1}
\nu\alpha_t
$$

$$
A_-(x)
=
\sum_{G_t(x)=-1}
\nu\alpha_t
$$

如果弱学习器序列完全一样，而且所有轮都乘同一个正数 $\nu$，最终分类方向不会因为 $\nu$ 变小而偏向某一类。

因为两边都同时乘了同一个正数：

$$
\nu>0
$$

真正发生变化的是下一轮训练。

### 学习率在样本权重更新里的体现

不加学习率时：

$$
w_i^{(t+1)}
=
\begin{cases}
\dfrac{w_i^{(t)} e^{-\alpha_t}}{Z_t},
& G_t(x_i)=y_i \\[10pt]
\dfrac{w_i^{(t)} e^{\alpha_t}}{Z_t},
& G_t(x_i)\ne y_i
\end{cases}
$$

加入学习率后：

$$
w_i^{(t+1)}
=
\begin{cases}
\dfrac{w_i^{(t)} e^{-\nu\alpha_t}}{Z_t},
& G_t(x_i)=y_i \\[10pt]
\dfrac{w_i^{(t)} e^{\nu\alpha_t}}{Z_t},
& G_t(x_i)\ne y_i
\end{cases}
$$

当 $\nu$ 比较小时：

$$
e^{\nu\alpha_t}
$$

不会特别大，

$$
e^{-\nu\alpha_t}
$$

也不会特别小。

所以：

$$
\text{分错样本权重上升得慢}
$$

$$
\text{分对样本权重下降得慢}
$$

说人话就是：

$$
\text{学习率越小，每一轮纠错力度越温和。}
$$

因此小学习率通常要配合更多弱学习器：

$$
\nu\text{ 小}
\Rightarrow
T\text{ 通常要大}
$$

也就是：

$$
\text{learning\_rate 小}
\Rightarrow
\text{n\_estimators 通常要大}
$$

In [ ]:
results = []

for n_estimators in [10, 50, 100, 200]:
    for learning_rate in [0.1, 0.5, 1.0]:
        try:
            model = AdaBoostClassifier(
                estimator=DecisionTreeClassifier(max_depth=1, random_state=22),
                n_estimators=n_estimators,
                learning_rate=learning_rate,
                random_state=22,
            )
        except TypeError:
            model = AdaBoostClassifier(
                base_estimator=DecisionTreeClassifier(max_depth=1, random_state=22),
                n_estimators=n_estimators,
                learning_rate=learning_rate,
                random_state=22,
            )

        model.fit(X_train, y_train)
        acc = accuracy_score(y_test, model.predict(X_test))
        results.append((n_estimators, learning_rate, acc))

results

观察结果时，不要机械记某个参数一定最好。

更重要的是建立感觉：

$$
\text{n\_estimators 控制总轮数}
$$

$$
\text{learning\_rate 控制每一轮的影响力度}
$$

如果学习率比较小：

$$
\text{每一轮走得慢}
$$

所以通常需要：

$$
\text{更多轮弱学习器}
$$

## 10. AdaBoost 的优缺点

优点：

- 思想清晰，适合理解 Boosting。
- 可以把很多弱学习器组合成强模型。
- 参数不算多。
- 不容易像单棵深树那样随意过拟合。

缺点：

- 对噪声和异常点比较敏感。
- 如果某些样本本身标注错了，它们会被反复加权。
- 实际工程中，GBDT、XGBoost、LightGBM 往往更常用。

为什么对噪声敏感？

```text
AdaBoost 会重点关注分错的样本。
如果这个样本本身就是错标或异常点，模型可能会被它牵着走。
```

## 11. Bagging、随机森林、AdaBoost 对比

| 方法 | 训练方式 | 重点解决 | 核心直觉 |
|---|---|---|---|
| Bagging | 并行 | 降低方差 | 多个模型看不同数据后投票 |
| 随机森林 | 并行 | 降低方差 | 样本随机 + 特征随机 + 多树投票 |
| AdaBoost | 串行 | 降低偏差 | 后面的模型重点修正前面的错误 |

简单说：

```text
Bagging 像让很多人独立做题再投票。
AdaBoost 像一个人不断整理错题，一个接一个补短板。
```

## 12. 本课小结

- AdaBoost 是 Boosting 的经典入门算法。
- 它按顺序训练多个弱学习器。
- 前一轮分错的样本，后一轮权重会变大。
- 表现好的弱学习器，在最终投票中权重更大。
- `n_estimators` 控制弱学习器数量。
- `learning_rate` 控制每个弱学习器的贡献大小。
- AdaBoost 对噪声和异常点比较敏感。

记忆方式：

```text
AdaBoost = 错题变重要 + 弱模型接力 + 加权投票。
```